In [0]:
# Read silver source tables for dimensions and facts
silver_shipment_current_df = spark.table("logistics.silver.silver_shipment_current")
delivery_attempts_df = spark.table("logistics.silver.delivery_attempts")
carrier_master_df = spark.table("logistics.silver.carrier_master")
warehouse_master_df = spark.table("logistics.silver.warehouse_master")
route_master_df = spark.table("logistics.silver.route_master")

for name, df in [
    ("silver_shipment_current", silver_shipment_current_df),
    ("delivery_attempts", delivery_attempts_df),
    ("carrier_master", carrier_master_df),
    ("warehouse_master", warehouse_master_df),
    ("route_master", route_master_df),
]:
    print(f"{name}: {df.count()} rows, {len(df.columns)} columns")

## Dimensional model (star) in logistics.gold

Two dimensions (`dim_carrier`, `dim_warehouse`) shared by three fact tables:

* **fact_shipment** (grain: one shipment) -> dim_carrier, dim_warehouse (origin)
* **fact_delivery_attempt** (grain: one delivery attempt) -> fact_shipment (shipment_sk), dim_carrier
* **fact_route** (grain: one route) -> dim_warehouse (origin); measures `distance_km`, `planned_hours`, `avg_speed_kmph`

Dates, service level, shipment status and driver are kept as columns on the facts.

Surrogate keys: every `*_sk` in the gold tables (dimensions and facts) is `sha2(natural_key, 256)`, a 64-character hex string. Composite keys are joined with `||` before hashing. The keys are deterministic, so the same natural key always gets the same key across rebuilds. Every dimension has an unknown member whose key is `sha2('UNKNOWN', 256)`; unmatched fact lookups map to it.

Known data gaps: shipment `origin_warehouse_id` values (ORIGIN_*) do not match `warehouse_master` (WAREHO*), so `origin_warehouse_sk` resolves to the unknown member. fact_route is not linked to fact_shipment because shipments carry no route_id, each destination city has several routes, and the shipment origin warehouse cannot be resolved.

In [0]:
import hashlib
from pyspark.sql import functions as F

GOLD = "logistics.gold"
# Unknown member key = sha2 of the literal 'UNKNOWN' (same value make_sk produces for that natural key)
UNKNOWN_SK = hashlib.sha256(b"UNKNOWN").hexdigest()

def make_sk(*nk_cols):
    """Hash-based surrogate key: SHA-256 of the natural key column(s), '||' separated."""
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in nk_cols]), 256)

def unknown_row(dim_df):
    """One-row DataFrame holding the unknown member for a dimension."""
    exprs = []
    for f in dim_df.schema.fields:  # schema read once per dimension
        t = f.dataType.simpleString()
        if f.name.endswith("_sk"):
            v = F.lit(UNKNOWN_SK)
        elif t == "string":
            v = F.lit("UNKNOWN")
        elif t == "date":
            v = F.lit("1900-01-01")
        elif t == "timestamp":
            v = F.lit("1900-01-01 00:00:00")
        elif t == "boolean":
            v = F.lit(False)
        else:
            v = F.lit(0)
        exprs.append(v.cast(f.dataType).alias(f.name))
    return spark.range(1).select(*exprs)

def build_dimension(src_df, sk_col, nk_cols, attr_cols):
    """Deduplicate on the natural key, assign a surrogate key, append the unknown member."""
    base = src_df.select(*nk_cols, *attr_cols).dropDuplicates(nk_cols)
    dim = (base
           .withColumn(sk_col, make_sk(*nk_cols))
           .select(sk_col, *nk_cols, *attr_cols))
    return dim.unionByName(unknown_row(dim)).withColumn("_gold_loaded_ts", F.current_timestamp())

def write_table(df, name):
    full = f"{GOLD}.{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full)
    print(f"{full}: {spark.table(full).count()} rows")

def sk_lookup(dim_name, nk_col, sk_col):
    """Natural key -> surrogate key map (unknown member excluded)."""
    return spark.table(f"{GOLD}.{dim_name}").filter(F.col(sk_col) != UNKNOWN_SK).select(nk_col, sk_col)

In [0]:
dim_carrier_df = build_dimension(
    carrier_master_df, "carrier_sk", ["carrier_id"],
    ["carrier_name", "service_region", "carrier_tier", "active_flag", "onboarded_date",
     "contract_type", "service_level", "country", "support_group"])
write_table(dim_carrier_df, "dim_carrier")

dim_warehouse_df = build_dimension(
    warehouse_master_df, "warehouse_sk", ["warehouse_id"],
    ["warehouse_name", "city", "region", "country", "capacity_units", "warehouse_type",
     "active_flag", "opening_date", "manager_code"])
write_table(dim_warehouse_df, "dim_warehouse")

In [0]:
attempt_agg_df = (delivery_attempts_df.groupBy("shipment_id")
    .agg(F.count("*").alias("attempt_count"),
         F.sum(F.when(F.col("attempt_result") == "FAILED", 1).otherwise(0)).alias("failed_attempt_count")))

wh_lookup = (sk_lookup("dim_warehouse", "warehouse_id", "warehouse_sk")
             .withColumnRenamed("warehouse_id", "origin_warehouse_id")
             .withColumnRenamed("warehouse_sk", "origin_warehouse_sk"))

fk_cols = ["carrier_sk", "origin_warehouse_sk"]

fact_shipment_df = (silver_shipment_current_df
    .join(F.broadcast(sk_lookup("dim_carrier", "carrier_id", "carrier_sk")), "carrier_id", "left")
    .join(F.broadcast(wh_lookup), "origin_warehouse_id", "left")
    .join(attempt_agg_df, "shipment_id", "left")
    .select(
        make_sk("shipment_id").alias("shipment_sk"),
        "shipment_id", "order_id", "customer_id",            # degenerate dimensions
        *[F.coalesce(c, F.lit(UNKNOWN_SK)).alias(c) for c in fk_cols],
        "origin_warehouse_id", "destination_city", "service_level", "shipment_status",
        "package_weight_kg", "shipping_cost",
        F.round((F.unix_timestamp("promised_delivery_ts") - F.unix_timestamp("created_ts")) / 3600, 2).alias("promised_lead_time_hours"),
        F.coalesce("attempt_count", F.lit(0)).cast("int").alias("attempt_count"),
        F.coalesce("failed_attempt_count", F.lit(0)).cast("int").alias("failed_attempt_count"),
        "created_ts", "promised_delivery_ts",
        F.current_timestamp().alias("_gold_loaded_ts"),
    ))

write_table(fact_shipment_df, "fact_shipment")

In [0]:
shipment_keys_df = spark.table(f"{GOLD}.fact_shipment").select("shipment_id", "shipment_sk", "carrier_sk")
attempt_fk_cols = ["shipment_sk", "carrier_sk"]

fact_delivery_attempt_df = (delivery_attempts_df
    .join(shipment_keys_df, "shipment_id", "left")
    .select(
        make_sk("attempt_id").alias("attempt_sk"),
        "attempt_id", "shipment_id", "driver_id",             # degenerate dimensions
        *[F.coalesce(c, F.lit(UNKNOWN_SK)).alias(c) for c in attempt_fk_cols],
        "attempt_sequence", "attempt_result", "failure_reason", "proof_of_delivery", "delivery_city",
        (F.col("attempt_result") == "FAILED").cast("int").alias("is_failed"),
        F.round((F.unix_timestamp("received_ts") - F.unix_timestamp("attempt_ts")) / 60, 2).alias("received_lag_minutes"),
        "attempt_ts", "received_ts",
        F.current_timestamp().alias("_gold_loaded_ts"),
    ))

write_table(fact_delivery_attempt_df, "fact_delivery_attempt")

In [0]:
# fact_route: grain = one route, linked to dim_warehouse through the origin warehouse
fact_route_df = (route_master_df
    .join(F.broadcast(wh_lookup), "origin_warehouse_id", "left")
    .select(
        make_sk("route_id").alias("route_sk"),
        "route_id",                                             # degenerate dimension
        F.coalesce("origin_warehouse_sk", F.lit(UNKNOWN_SK)).alias("origin_warehouse_sk"),
        "origin_warehouse_id", "destination_city", "route_class", "vehicle_class",
        "service_region", "route_owner", "active_flag",
        "distance_km", "planned_hours",
        F.round(F.col("distance_km") / F.nullif(F.col("planned_hours"), F.lit(0)), 2).alias("avg_speed_kmph"),
        F.current_timestamp().alias("_gold_loaded_ts"),
    ))

write_table(fact_route_df, "fact_route")

In [0]:
dims = {
    "dim_carrier": ("carrier_sk", "carrier_id"),
    "dim_warehouse": ("warehouse_sk", "warehouse_id"),
}
checks = []
for d, (sk, nk) in dims.items():
    r = spark.table(f"{GOLD}.{d}").agg(F.count("*").alias("n"), F.countDistinct(sk).alias("sk"),
                                       F.countDistinct(nk).alias("nk"),
                                       F.sum((F.col(sk) == UNKNOWN_SK).cast("int")).alias("unk")).first()
    checks.append((f"{d} key uniqueness", r.n == r.sk == r.nk and r.unk == 1, f"rows={r.n} distinct_sk={r.sk} distinct_nk={r.nk} unknown_rows={r.unk}"))

# Key format: every *_sk column in every gold table is a SHA-256 hex string
for t in [*dims, "fact_shipment", "fact_delivery_attempt", "fact_route"]:
    tdf = spark.table(f"{GOLD}.{t}")
    sk_cols = [c for c in tdf.columns if c.endswith("_sk")]
    bad = tdf.filter(" OR ".join(f"NOT ({c} RLIKE '^[0-9a-f]{{64}}$') OR {c} IS NULL" for c in sk_cols)).count()
    checks.append((f"{t} sha2 key format", bad == 0, f"columns={sk_cols} invalid_rows={bad}"))

# Referential integrity: every fact FK must exist in its parent table
fks = [
    ("fact_shipment", "carrier_sk", "dim_carrier", "carrier_sk"),
    ("fact_shipment", "origin_warehouse_sk", "dim_warehouse", "warehouse_sk"),
    ("fact_delivery_attempt", "carrier_sk", "dim_carrier", "carrier_sk"),
    ("fact_route", "origin_warehouse_sk", "dim_warehouse", "warehouse_sk"),
]
for child, fk, parent, pk in fks:
    c = spark.table(f"{GOLD}.{child}")
    orphans = c.join(spark.table(f"{GOLD}.{parent}").select(F.col(pk).alias(fk)), fk, "left_anti").count()
    unknown = c.filter(F.col(fk) == UNKNOWN_SK).count()
    checks.append((f"{child}.{fk} -> {parent}", orphans == 0, f"orphans={orphans} mapped_to_unknown={unknown}"))

fs = spark.table(f"{GOLD}.fact_shipment")
fa = spark.table(f"{GOLD}.fact_delivery_attempt")
ship_orphans = fa.filter(F.col("shipment_sk") != UNKNOWN_SK).join(fs.select("shipment_sk"), "shipment_sk", "left_anti").count()
checks.append(("fact_delivery_attempt.shipment_sk -> fact_shipment", ship_orphans == 0,
               f"orphans={ship_orphans} mapped_to_unknown={fa.filter(F.col('shipment_sk') == UNKNOWN_SK).count()}"))

# Reconciliation with silver
src = silver_shipment_current_df.agg(F.count("*").alias("n"), F.round(F.sum("shipping_cost"), 2).alias("cost")).first()
tgt = fs.agg(F.count("*").alias("n"), F.countDistinct("shipment_sk").alias("sk"), F.round(F.sum("shipping_cost"), 2).alias("cost"), F.sum("attempt_count").alias("att")).first()
n_attempts = delivery_attempts_df.count()
checks.append(("fact_shipment rows/cost vs silver", src.n == tgt.n == tgt.sk and src.cost == tgt.cost, f"silver={src.n}/{src.cost} gold={tgt.n}/{tgt.cost}"))
checks.append(("fact_delivery_attempt rows vs silver", fa.count() == n_attempts, f"silver={n_attempts} gold={fa.count()}"))

fr = spark.table(f"{GOLD}.fact_route").agg(F.count("*").alias("n"), F.countDistinct("route_sk").alias("sk"), F.round(F.sum("distance_km"), 2).alias("km")).first()
sr = route_master_df.agg(F.count("*").alias("n"), F.round(F.sum("distance_km"), 2).alias("km")).first()
checks.append(("fact_route rows/distance vs silver", sr.n == fr.n == fr.sk and sr.km == fr.km, f"silver={sr.n}/{sr.km} gold={fr.n}/{fr.km}"))
checks.append(("fact_shipment.attempt_count vs attempts linked", tgt.att == fa.filter(F.col("shipment_sk") != UNKNOWN_SK).count(), f"sum_attempt_count={tgt.att}"))

display(spark.createDataFrame(checks, "check string, passed boolean, detail string"))